# Etapa 3 — Comparação dos modelos (re-run BLINDADO)

Refaz só a tabela comparativa, com 3 proteções novas:
- **Salva o CSV a cada modelo** (se cair, mantém o que já rodou)
- **Gradient clipping** (conserta o `nan` do VGG16 fine-tuning)
- **Acha a pasta do repo sozinho** (resolve o bug do nome `Trabalho_IA`)

Pula a varredura otimizador×LR: a melhor config (SGD, lr=0.01) já foi determinada antes.

Configure: **GPU On**, **Internet On**, **Add Input** com o `pathmnist_224.npz`. Faça **Save Version → Save & Run All** para não perder os resultados.

## 1. Setup

In [ ]:
import os, sys, glob, torch
%cd /kaggle/working
REPO_URL = 'https://github.com/SEU_USUARIO/SEU_REPO.git'  # <-- EDITE
hit = glob.glob('/kaggle/working/**/src/utils/seeds.py', recursive=True)
if not hit:
    !git clone $REPO_URL
    hit = glob.glob('/kaggle/working/**/src/utils/seeds.py', recursive=True)
ROOT = hit[0].split('/src/')[0]
os.chdir(ROOT); sys.path.insert(0, ROOT)
print('raiz:', ROOT, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!pip install -q medmnist

## 2. DataLoaders

In [ ]:
from src.utils.seeds import set_seed
from src.etapa2_pytorch.data_224 import make_loaders
set_seed(42)
DATA_ROOT = os.path.dirname(glob.glob('/kaggle/input/**/pathmnist_224.npz', recursive=True)[0])
train_loader, val_loader, test_loader = make_loaders(size=224, batch_size=64, num_workers=2, root=DATA_ROOT)
print('batches treino:', len(train_loader))

## 3. Comparação dos 9 modelos/modos
SGD lr=0.01 (melhor config). `epochs=5` (baixe para 3 se quiser mais rápido). Salva `results/logs/etapa3_comparacao_modelos.csv` a cada modelo concluído.

In [ ]:
from src.etapa3_cnn_vit.run_experiments import compare_models
resultados = compare_models(train_loader, val_loader,
                            optimizer='sgd', lr=0.01, epochs=5, pretrained=True)
import pandas as pd; pd.DataFrame(resultados)

## 4. Conferir a tabela salva

In [ ]:
import pandas as pd
df = pd.read_csv('results/logs/etapa3_comparacao_modelos.csv')
print(df.to_string(index=False))

## 5. Salvar
**Save Version → Save & Run All**. Depois baixe `results/logs/etapa3_comparacao_modelos.csv` e os checkpoints de `results/checkpoints/` para o repositório.